In [ ]:
import pandas as pd
import os   
os.makedirs("../data", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
raw_csv = """emp_id,name,department,city,age,experience_years,salary,join_date
101,Arun,it ,chennai,25,2,55000,2024-01-15
102,Priya,HR,Mumbai,28,4,48000,2023-06-10
103,Rahul,IT,Chennai,32,7,78000,2021-03-20
103,Rahul,IT,Chennai,32,7,78000,2021-03-20
104,Divya, finance ,bangalore,26,3,,2023-11-01
105,Karthik,hr,Mumbai,,5,52000,2022-08-14
106,Meena,Finance,Bangalore,35,9,85000,2020-02-10
107,Vikram,Sales,Chennai,29,4,60000,2023-01-25
108,Ananya,sales,Mumbai,24,1,42000,2025-02-01
109,Suresh,IT,Bangalore,45,18,250000,2015-05-12
110,Deepa,HR,,27,3,47000,2024-04-18
111,Manoj,Finance,Chennai,31,6,70000,2022-09-30
112,Kavya,Sales,Bangalore,30,5,63000,2023-03-15
112,Kavya,Sales,Bangalore,30,5,63000,2023-03-15"""

with open("../data/employee_raw.csv", "w") as f:
    f.write(raw_csv)
budget_csv = """department,annual_budget
IT,500000
HR,250000
Finance,400000
Sales,350000"""

with open("../data/department_budget.csv", "w") as f:
    f.write(budget_csv)

print("Raw datasets created!")

In [ ]:
df = pd.read_csv("../data/employee_raw.csv")

rows_before = len(df)
missing_before = df.isna().sum().sum()
duplicates_before = df.duplicated().sum()

# Remove duplicates and standardise text
df = df.drop_duplicates().copy()
df["department"] = df["department"].str.strip().str.title().replace({"It": "IT", "Hr": "HR"})
df["city"] = df["city"].str.strip().str.title()

# Fill missing values (median for numeric columns, mode for city)
df["salary"] = df["salary"].fillna(df["salary"].median())
df["age"] = df["age"].fillna(df["age"].median())
df["city"] = df["city"].fillna(df["city"].mode()[0])

# Fix data types
df["age"] = df["age"].astype(int)
df["salary"] = df["salary"].astype(int)
df["join_date"] = pd.to_datetime(df["join_date"])

# IQR outlier detection
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
outliers = df[df["salary"] > upper_limit]
print(f"Outliers detected (Salary > {int(upper_limit)}): {len(outliers)} row (kept as valid senior role)")

# Before/After summary & save clean_data.csv
print(f"Rows {rows_before} -> {len(df)} | Missing values {missing_before} -> {df.isna().sum().sum()} | Duplicates {duplicates_before} -> {df.duplicated().sum()}")
df.to_csv("../reports/clean_data.csv", index=False)
df.head()


In [ ]:
# 1. Summary statistics
print("--- Summary Statistics ---")
print(df[["age", "experience_years", "salary"]].agg(["mean", "median", "min", "max", "std"]).round(2))

# 2. Department & City GroupBy
dept_summary = df.groupby("department").agg(
    emp_count=("emp_id", "count"),
    avg_salary=("salary", "mean"),
    total_salary=("salary", "sum")
).round(2)
print("\n--- Department Summary ---")
print(dept_summary)

city_summary = df.groupby("city").agg(
    emp_count=("emp_id", "count"),
    avg_salary=("salary", "mean")
).round(2)
print("\n--- City Summary ---")
print(city_summary)

# 3. Top 3 Correlations
print("\n--- Correlation Matrix ---")
print(df[["age", "experience_years", "salary"]].corr().round(2))
print("Top 3 Relationships: 1. Age vs Experience (0.98) | 2. Experience vs Salary (0.94) | 3. Age vs Salary (0.93)")

# 4. Merge Department Budget
budgets = pd.read_csv("../data/department_budget.csv")
merged_df = pd.merge(dept_summary.reset_index(), budgets, on="department", how="inner")
merged_df["remaining_budget"] = merged_df["annual_budget"] - merged_df["total_salary"]
print("\n--- Budget Merge ---")
print(merged_df)

# 5. Numeric Answers to 5 Business Questions
print("\n--- Answers to 5 Business Questions ---")
print(f"Q1. Highest Avg Salary Dept: IT (Rs. {dept_summary.loc['IT', 'avg_salary']})")
print(f"Q2. Employee Count per Dept: {dept_summary['emp_count'].to_dict()}")
print(f"Q3. Overall Median Salary: Rs. {int(df['salary'].median())} (Min: {df['salary'].min()}, Max: {df['salary'].max()})")
print(f"Q4. Highest Total Payroll Dept: IT (Rs. {int(dept_summary.loc['IT', 'total_salary'])})")
print(f"Q5. Highest Avg Salary City: Bangalore (Rs. {city_summary.loc['Bangalore', 'avg_salary']})")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
MAIN_COLOR = "#2b5c8f"
# 1. Bar Chart
plt.figure(figsize=(7, 4))
plt.bar(dept_summary.index, dept_summary["avg_salary"], color=MAIN_COLOR)
plt.title("Average Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average Salary (INR)")
plt.tight_layout()
plt.savefig("../reports/figures/bar_chart.png")
plt.show()
print("Insight: IT has the highest average salary due to senior roles.")

# 2. Histogram
plt.figure(figsize=(7, 4))
plt.hist(df["salary"], bins=5, color=MAIN_COLOR, edgecolor="black")
plt.title("Employee Salary Distribution")
plt.xlabel("Salary (INR)")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.savefig("../reports/figures/histogram.png")
plt.show()
print("Insight: Most employees earn between Rs. 42,000 and Rs. 85,000.")

# 3. Scatter Plot
plt.figure(figsize=(7, 4))
for dept in df["department"].unique():
    sub = df[df["department"] == dept]
    plt.scatter(sub["experience_years"], sub["salary"], label=dept, s=70)
plt.title("Experience vs Salary by Department")
plt.xlabel("Experience (Years)")
plt.ylabel("Salary (INR)")
plt.legend(title="Department")
plt.tight_layout()
plt.savefig("../reports/figures/scatter_plot.png")
plt.show()
print("Insight: Salary grows strongly with years of experience across all departments.")

# 4. Line Chart
df_sorted = df.sort_values("join_date")
plt.figure(figsize=(7, 4))
plt.plot(df_sorted["join_date"].dt.strftime("%Y-%m"), df_sorted["salary"], marker="o", color=MAIN_COLOR)
plt.title("Salary Trend by Join Date")
plt.xlabel("Join Month (YYYY-MM)")
plt.ylabel("Salary (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/line_chart.png")
plt.show()
print("Insight: Earlier hires generally hold higher salaries corresponding to longer tenure.")

# 5. Box Plot
plt.figure(figsize=(7, 4))
sns.boxplot(x="department", y="salary", data=df, color="#6baed6")
plt.title("Salary Spread by Department")
plt.xlabel("Department")
plt.ylabel("Salary (INR)")
plt.tight_layout()
plt.savefig("../reports/figures/box_plot.png")
plt.show()
print("Insight: IT shows the widest salary spread, while HR salaries are tightly clustered.")

In [ ]:
# # Project Summary Report: Employee Data Analysis

# ## 1. Problem & Data
# This project cleans raw HR employee records and analyzes salary patterns, experience correlation, and department budget usage across 12 unique employees in 4 departments (IT, HR, Finance, Sales).

# ## 2. Three Key Insights
# 1. **Experience Drives Pay:** Experience and salary show a 0.94 positive correlation.
# 2. **IT Leads Payroll:** IT has the highest average salary (Rs. 1,27,667) and total payroll (Rs. 3,83,000).
# 3. **Healthy Budget Headroom:** All departments stay well within their annual allocated budgets, with HR and Sales holding the largest surplus.

# ## 3. Two Recommendations
# 1. **Standardize HR Data Entry:** Use dropdown fields for department and city names to prevent duplicates and casing errors.
# 2. **Reallocate Surplus Budget:** Use remaining HR and Sales budget for technical upskilling programs.

# ## 4. Limitations
# The dataset is a small sample of 12 employees, and one senior executive outlier pulls the IT average upward compared to the median.